# Module 3 Demo: SLAM Data Doctor — 漂移趋势预测

本 Notebook 使用**合成数据**模拟双目外参漂移场景，演示极线约束监控与趋势预测。

In [ ]:
import sys
sys.path.insert(0, '../03_Data_Doctor_for_SLAM')

import numpy as np
import matplotlib.pyplot as plt
from data_doctor import EpipolarDriftMonitor, DriftTrendPredictor, EpipolarDriftReport

## 1. 模拟极线误差序列

三种场景：
- **稳定运行**（前 50 帧）：误差 ≈ 0.5px，随机波动
- **热漂移**（50-80 帧）：误差缓慢上升
- **碰撞跳变**（80 帧后）：误差阶跃至 4px

In [ ]:
np.random.seed(42)
n_frames = 100

errors = np.zeros(n_frames)
for i in range(n_frames):
    if i < 50:
        # 稳定运行
        errors[i] = 0.5 + np.random.normal(0, 0.15)
    elif i < 80:
        # 热漂移：线性增长
        errors[i] = 0.5 + (i - 50) * 0.06 + np.random.normal(0, 0.2)
    else:
        # 碰撞跳变
        errors[i] = 4.0 + np.random.normal(0, 0.3)

plt.figure(figsize=(12, 4))
plt.plot(errors, 'b.-', markersize=3)
plt.axhline(y=2.0, color='orange', linestyle='--', label='Drift threshold (2px)')
plt.axhline(y=3.0, color='red', linestyle='--', label='Critical threshold (3px)')
plt.axvspan(0, 50, alpha=0.1, color='green', label='Stable')
plt.axvspan(50, 80, alpha=0.1, color='orange', label='Thermal drift')
plt.axvspan(80, 100, alpha=0.1, color='red', label='Post-collision')
plt.xlabel('Frame')
plt.ylabel('Mean Epipolar Error [px]')
plt.title('Simulated Stereo Extrinsics Drift')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 2. 滑动窗口趋势预测

In [ ]:
predictor = DriftTrendPredictor(window_size=20)

trends = []
for i in range(n_frames):
    # 构造模拟报告
    report = EpipolarDriftReport(
        frame_id=i,
        mean_epipolar_error=errors[i],
        max_epipolar_error=errors[i] * 1.5,
        outlier_ratio=0.1,
        drift_detected=errors[i] > 2.0
    )
    trend = predictor.update(report)
    trends.append(trend)

# 绘制趋势
window_means = [t.window_mean for t in trends]
window_slopes = [t.window_trend for t in trends]
predicted = [t.predicted_error for t in trends]
statuses = [t.trend_status for t in trends]

fig, (ax1, ax2, ax3) = plt.subplots(3, 1, figsize=(12, 10), sharex=True)

# 误差 + 预测
ax1.plot(errors, 'b.-', markersize=3, label='Actual error')
ax1.plot(predicted, 'r--', linewidth=1.5, label='Predicted (next frame)')
ax1.axhline(y=2.0, color='orange', linestyle=':', alpha=0.7)
ax1.set_ylabel('Epipolar Error [px]')
ax1.set_title('Drift Trend Prediction')
ax1.legend()
ax1.grid(True, alpha=0.3)

# 趋势斜率
ax2.plot(window_slopes, 'g-', linewidth=1.5)
ax2.axhline(y=0.05, color='orange', linestyle='--', label='Drifting threshold')
ax2.axhline(y=0, color='gray', linestyle=':')
ax2.set_ylabel('Trend Slope [px/frame]')
ax2.legend()
ax2.grid(True, alpha=0.3)

# 状态
status_map = {'STABLE': 0, 'DRIFTING': 1, 'CRITICAL': 2}
status_vals = [status_map.get(s, 0) for s in statuses]
colors = ['green' if v==0 else 'orange' if v==1 else 'red' for v in status_vals]
ax3.scatter(range(n_frames), status_vals, c=colors, s=10)
ax3.set_yticks([0, 1, 2])
ax3.set_yticklabels(['STABLE', 'DRIFTING', 'CRITICAL'])
ax3.set_xlabel('Frame')
ax3.set_ylabel('Status')
ax3.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()